In [1]:
!pip install --upgrade pip --quiet
!pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn plotly scipy statsmodels imbalanced-learn ipykernel lightgbm openpyxl --quiet

# Concept Reference — Embargo in Rolling-Window Cross-Validation

## What Problem Does Embargo Solve?

This notebook predicts **1-year and 3-year forward returns** — meaning the *target* for day 1 is "what did the stock return from day 1 to day 252?"

Now imagine your training window ends on day 1000, and your test window starts on day 1001.

The target for day 999 (in your training set) includes return data from days 999 → 1251.  
But your test window starts at day 1001 — **which is inside that future return window**.  
So the model has indirectly "seen" data from the test period while training. It's like studying from the exam paper itself.

This is called **data leakage** and it inflates CV scores in backtests but fails badly in live trading.

---

## What Embargo Does

An embargo is a **mandatory gap between the end of training and the start of testing**.

```
[ Training Data ]  ...gap (embargo)...  [ Test Data ]
     ends day 1000     252 days gap         starts day 1252
```

For a 1-year (252-day) horizon, embargo = 252 days.  
This ensures the last training sample's forward-return window **fully closes** before the test set begins.

In code (Cell 4 — `rolling_window_folds`):
```python
test_start = train_end + embargo_days   # <-- the gap
```

And in Cell 7, the embargo is set dynamically to match the horizon:
```python
rolling_window_folds(len(d), embargo_days=horizon_days)
# horizon_days = 252 for 1Y, 756 for 3Y
```

For the **3-year horizon**, the embargo is 756 days (~3 years) — which is why only 9 folds are produced instead of 13; many potential folds get skipped because there isn't enough data left after the gap.

---

## Visual Timeline

```
|<--- 5yr Train --->|<-- embargo (= horizon) -->|<-- 1yr Test -->|
    days 0–1260          days 1260–1512              days 1512–1764
```

---

## Alternatives to Embargo

| Alternative | What It Does | Tradeoff |
|---|---|---|
| **Purging** | Remove specific rows from training whose target windows overlap with the test period | More surgical than a fixed gap; common in financial ML (López de Prado's MLFS book) |
| **Hard cutoff** | Never use data past a certain date for training | Simple but wastes a lot of data |
| **Non-overlapping targets** | Predict next-day return instead of 1-year cumulative (no overlap possible) | Avoids the problem entirely but loses long-horizon signal |
| **Gap sampling** | Skip every Nth row so consecutive samples don't have overlapping windows | Reduces leakage but doesn't eliminate it |
| **Combinatorial Purged CV (CPCV)** | Advanced version — purges overlapping samples across all fold combinations | Gold standard for financial ML, but computationally expensive |

---

## Quick Reference Table

| Term | Meaning |
|---|---|
| **Leakage** | Model accidentally trained on future data |
| **Embargo** | A time gap that prevents this overlap |
| **Horizon** | How far into the future your target looks |
| **Rule of thumb** | Embargo length ≥ prediction horizon length |

> **Key tradeoff**: Bigger horizon → bigger embargo → fewer usable folds → less reliable CV estimates. This is why the 3Y horizon has fewer folds than the 1Y horizon in this notebook.

In [2]:
"""
Aurora Financial Markets Module — Step 3  (v5.1 — Dual-Horizon + Live Tail Fix)
ML-Driven Portfolio Strategy: Prediction, Optimization & Backtest
================================================================
Data   : Market_Data_Revised.xlsx (6 Indian stocks, 2005–2026)
         + Macroeconomic_QRT  (quarterly GDP growth, Inflation)
         + Macroeconomic_Annual (annual GDP growth, Inflation)

Ensemble: XGBoost + LightGBM + Random Forest
"""

import pandas as pd
import numpy as np
import warnings
import scipy.optimize as sco
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

FILE          = "sample_data/Market_Data_Revised.xlsx"
RF_RATE       = 0.065    # Indian risk-free rate (~6.5% p.a.)
TRAIN_DAYS    = 1260     # Rolling training window  (~5 years)
TEST_DAYS     = 252      # Test window per fold     (~1 year)
BACKTEST_DAYS = 504      # Final backtest window    (~2 years)
EMBARGO_DAYS  = 63       # 1 quarter — prevents leakage for long-horizon targets
N_FOLDS       = 14       # Walk-forward folds anchored from 2005

# ── Prediction horizons ────────────────────────────────────────────
HORIZON_1Y = 252    # 1-year forward return (trading days)
HORIZON_3Y = 756    # 3-year forward return (trading days)

# ── BUY signal thresholds (must beat risk-free to be BUY) ─────────
BUY_THRESHOLD_1Y = RF_RATE        # annualised, e.g. 6.5%
BUY_THRESHOLD_3Y = RF_RATE        # annualised, e.g. 6.5%

# ── Model hyperparameters ──────────────────────────────────────────
XGB_PARAMS = dict(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbosity=0, eval_metric="rmse",
)

LGBM_PARAMS = dict(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    num_leaves=31, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, verbosity=-1, n_jobs=-1,
)

RF_PARAMS = dict(
    n_estimators=200, max_depth=5, max_features="sqrt",
    min_samples_leaf=5, random_state=42, n_jobs=-1,
)

# ── Feature list ──────────────────────────────────────────────────
FEATURES = [
    # Short-horizon price/volume signals
    "ret_1", "ret_5", "ret_20",
    "vol_5", "vol_20",
    "mom_5", "mom_20",
    "vol_ratio", "price_range",
    # Long-horizon trend/context signals
    "mom_252", "mom_756",
    "price_vs_ma252", "price_vs_ma756",
    "vol_252", "vol_ratio_sr_lr", "vol_ratio_252",
    # Macroeconomic features
    "macro_gdp_qrt", "macro_inflation_qrt",
    "macro_gdp_ann",  "macro_inflation_ann",
]

MACRO_FEATURES = {
    "macro_gdp_qrt", "macro_inflation_qrt",
    "macro_gdp_ann",  "macro_inflation_ann",
}

LONG_FEATURES = {
    "mom_252", "mom_756",
    "price_vs_ma252", "price_vs_ma756",
    "vol_252", "vol_ratio_sr_lr", "vol_ratio_252",
}

MODEL_NAMES = ["XGBoost", "LightGBM", "RandomForest"]
SEP         = "─" * 110



In [3]:
# ─────────────────────────────────────────────
# 1. LOAD MACROECONOMIC DATA
# ─────────────────────────────────────────────

def load_macro(file: str) -> pd.DataFrame:
    qrt = pd.read_excel(file, sheet_name="Macroeconomic_QRT", usecols=[0, 1, 2])
    qrt.columns = ["date", "macro_gdp_qrt", "macro_inflation_qrt"]
    qrt = qrt.dropna(subset=["date"])
    qrt["date"] = pd.to_datetime(qrt["date"])
    qrt = qrt.set_index("date").sort_index()

    ann = pd.read_excel(file, sheet_name="Macroeconomic_Annual", usecols=[0, 1, 2])
    ann.columns = ["year", "macro_gdp_ann", "macro_inflation_ann"]
    ann = ann.dropna(subset=["year"])
    ann["date"] = pd.to_datetime(ann["year"].astype(str) + "-01-01")
    ann = ann.set_index("date")[["macro_gdp_ann", "macro_inflation_ann"]].sort_index()

    full_start = min(qrt.index.min(), ann.index.min())
    full_end   = max(qrt.index.max(), ann.index.max()) + pd.offsets.YearEnd(1)
    daily_idx  = pd.date_range(full_start, full_end, freq="D")

    qrt_daily   = qrt.reindex(daily_idx).ffill()
    ann_daily   = ann.reindex(daily_idx).ffill()
    macro_daily = pd.concat([qrt_daily, ann_daily], axis=1)
    return macro_daily


# ─────────────────────────────────────────────
# 2. LOAD STOCK DATA
# ─────────────────────────────────────────────

macro_daily = load_macro(FILE)

print("Macroeconomic data loaded:")
print(f"  Quarterly → {macro_daily['macro_gdp_qrt'].dropna().index[0].date()} "
      f"to {macro_daily['macro_gdp_qrt'].dropna().index[-1].date()}")
print(f"  Annual    → {macro_daily['macro_gdp_ann'].dropna().index[0].date()} "
      f"to {macro_daily['macro_gdp_ann'].dropna().index[-1].date()}")


Macroeconomic data loaded:
  Quarterly → 2005-01-01 to 2025-12-31
  Annual    → 2005-01-01 to 2025-12-31


In [4]:
dfs        = {}
names_list = []

for i in range(1, 7):
    df   = pd.read_excel(FILE, sheet_name=f"Comp {i}")
    name = df["company_name"].iloc[0].strip()
    df   = df.sort_values("date").reset_index(drop=True)
    dfs[name] = df
    names_list.append(name)

dfs.keys()

dict_keys(['ASIAN PAINTS LTD.', 'CIPLA LTD.', 'H D F C BANK LTD.', 'I T C LTD.', 'TATA CONSULTANCY SERVICES LTD.', 'ULTRATECH CEMENT LTD.'])

The expression `d["closing"].shift(5).bfill()`.

Let's break it into two steps.

**Step 1 — what shift(5) produces**

`shift(5)` moves all values down by 5 rows, creating NaN in the first 5 positions because there is no price from 5 days before those early rows.

Example closing prices:

| Row | closing | after shift(5) |
|-----|---------|----------------|
| 0   | 100     | NaN            |
| 1   | 102     | NaN            |
| 2   | 101     | NaN            |
| 3   | 103     | NaN            |
| 4   | 104     | NaN            |
| 5   | 106     | 100            |
| 6   | 108     | 102            |
| 7   | 110     | 101            |
| 7   | 110     | 101            |

The first 5 rows are NaN because there is no row -5, -4, -3, -2, -1.

---

**Step 2 — what bfill() fills into those NaNs**

`bfill()` = **backward fill**. It looks forward in the column and fills each NaN with the **next available non-NaN value below it**.

Applying bfill to the shifted column:

| Row | after shift(5) | after bfill() |
|-----|----------------|---------------|
| 0   | NaN            | **100** ← taken from Row 5 |
| 1   | NaN            | **100** ← taken from Row 5 |
| 2   | NaN            | **100** ← taken from Row 5 |
| 3   | NaN            | **100** ← taken from Row 5 |
| 4   | NaN            | **100** ← taken from Row 5 |
| 5   | 100            | 100            |
| 6   | 102            | 102            |
| 7   | 101            | 101            |

So rows 0–4 all get filled with 100 (the first real value that appears at row 5).

---

**What mom_5 becomes for those early rows**

After bfill, the momentum calculation becomes:

$$
mom\_5(t) = \frac{P_t}{\text{bfilled value}} - 1
$$

For row 0: $\frac{100}{100} - 1 = 0\%$  
For row 1: $\frac{102}{100} - 1 = 2\%$  
For row 2: $\frac{101}{100} - 1 = 1\%$  

These are **not genuine 5-day momentum values** — they are artifacts produced by using a future price (row 5's value = 100) to fill early missing denominators.

---

**Why this is a subtle information leak**

The price 100 at row 5 is a *future value* relative to rows 0–4. Using it in the denominator means those early rows implicitly "know" about a future price. This is a backward-fill leak.

**In practice the impact is small** because:
- The notebook drops the first few rows with `dropna(subset=FEATURES)`, which tends to clean up most of these artificial early values.
- Momentum signals from 2005 or 2006 (the very start of a 20-year dataset) have minimal weight in training.

**The clean alternative** is simply not using `bfill()`:
```python
d["mom_5"] = (d["closing"] / d["closing"].shift(5) - 1).shift(1)
```
This produces NaN for the first 5 rows, which then get dropped naturally by the later `dropna` call — no leakage at all.

In [5]:
# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING  (Expanding windows + Live Tail)
# ─────────────────────────────────────────────

def make_features(df: pd.DataFrame, macro: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values("date").reset_index(drop=True)

    # ── Short-horizon features ──────────────────────────────────────────
    d["ret_1"]       = d["Return"].shift(1)
    d["ret_5"]       = d["Return"].rolling(5, min_periods=1).mean().shift(1) # .rolling(5, min_periods=1). This tells pandas to look at a moving window of the last 5 rows.
    d["ret_20"]      = d["Return"].rolling(20, min_periods=1).mean().shift(1)
    d["vol_5"]       = d["Return"].rolling(5, min_periods=1).std().shift(1) # This is a measure of recent short-term volatility for last 5 days. if data exists for less than 5 days, it uses whatever is available (min_periods=1).
    d["vol_20"]      = d["Return"].rolling(20, min_periods=1).std().shift(1) # This is a measure of recent medium-term volatility for last 20 days.
    
    """ 
    .rolling() computes rolling statistics (like mean, std) over a specified window size.
    .shift(1) shifts the data down by 1 row, so that the feature values at time t are based on data up to time t-1, preventing look-ahead bias
    .bfill() fills any resulting NaN values (from the shift) with the next valid value, ensuring we don't lose rows at the start of the dataset.
    
    Short-term price momentum over about 1 week of trading.
    Positive value means price is higher than 5 trading days ago.
    Negative value means price is lower than 5 trading days ago. """

    d["mom_5"]       = (d["closing"] / d["closing"].shift(5).bfill()  - 1).shift(1)
    d["mom_20"]      = (d["closing"] / d["closing"].shift(20).bfill() - 1).shift(1)
    d["vol_ratio"]   = (d["trade Volume"] / d["trade Volume"].rolling(20, min_periods=1).mean()).shift(1)
    d["price_range"] = ((d["high"] - d["low"]) / d["closing"]).shift(1)

    # ── Long-horizon features (Expanding windows for 2005-2008) ─────────
    d["mom_252"]  = (d["closing"] / d["closing"].shift(252).bfill() - 1).shift(1)
    d["mom_756"]  = (d["closing"] / d["closing"].shift(756).bfill() - 1).shift(1)

    # price distance from 1Y moving average and 3Y moving average
    # These are regime/context features, not direct return features. 
    # They tell the model where the stock sits relative to its own long history.
    
    d["price_vs_ma252"] = (d["closing"] / d["closing"].rolling(252, min_periods=1).mean() - 1).shift(1)
    d["price_vs_ma756"] = (d["closing"] / d["closing"].rolling(756, min_periods=1).mean() - 1).shift(1)

    d["vol_252"]         = d["Return"].rolling(252, min_periods=1).std().shift(1)
    d["vol_ratio_sr_lr"] = (d["vol_20"] / d["vol_252"]).shift(1) #vol_ratio and vol_ratio_252: abnormal trading activity relative to normal baseline
    d["vol_ratio_252"]   = (d["trade Volume"] / d["trade Volume"].rolling(252, min_periods=1).mean()).shift(1)

    # ── Macro join ──────────────────────────────────────────────────────
    d = d.set_index("date")
    macro_aligned = macro.reindex(d.index, method="ffill")
    d["macro_gdp_qrt"]       = macro_aligned["macro_gdp_qrt"].shift(1)
    d["macro_inflation_qrt"] = macro_aligned["macro_inflation_qrt"].shift(1)
    d["macro_gdp_ann"]       = macro_aligned["macro_gdp_ann"].shift(1)
    d["macro_inflation_ann"] = macro_aligned["macro_inflation_ann"].shift(1)
    d = d.reset_index()

    # ── Forward return targets ──────────────────────────────────────────

    """1 + d["Return"]
    Turns daily return into daily growth factor.
    Example: if daily return is 2%, factor is 1.02. If -1%, factor is 0.99.

    rolling(HORIZON_1Y)
    Builds a moving window of length 252 trading days (for 1Y).

    apply(np.prod, raw=True)
    Multiplies all 252 daily factors in the window.
    That gives cumulative growth over the full window.

    1
    Converts cumulative growth back to cumulative return.
    If product is 1.18, return is 0.18 (18%).
    shift(-HORIZON_1Y)
    Moves that computed 252-day return upward by 252 rows so that row t gets the return from t to t+252.
    This is why it is a forward target label."""
    
    d["fwd_ret_1y"] = ((1 + d["Return"]).rolling(HORIZON_1Y).apply(np.prod, raw=True) - 1).shift(-HORIZON_1Y)
    d["fwd_ret_3y"] = ((1 + d["Return"]).rolling(HORIZON_3Y).apply(np.prod, raw=True) - 1).shift(-HORIZON_3Y)

    # 1. Neutralize toxic infinity values (caused by division by zero, e.g., zero volume)
    d = d.replace([np.inf, -np.inf], np.nan)

    # 2. Safely drop rows missing features (removes the first ~2 days, and any inf errors)
    #    CRITICAL: Does NOT drop the 2023-2026 tail where only the targets are NaN!
    d = d.dropna(subset=FEATURES).reset_index(drop=True)

    # Re-insert NaNs for the tail targets (fillna(0) affected them above)
    d.loc[d.index[-HORIZON_1Y:], "fwd_ret_1y"] = np.nan
    d.loc[d.index[-HORIZON_3Y:], "fwd_ret_3y"] = np.nan

    return d



In [6]:
# ─────────────────────────────────────────────
# 4. ROLLING-WINDOW CV FOLD BUILDER
# ─────────────────────────────────────────────

def rolling_window_folds(n_samples:    int,
                         train_days:   int = TRAIN_DAYS,
                         test_days:    int = TEST_DAYS,
                         embargo_days: int = EMBARGO_DAYS,
                         n_folds:      int = N_FOLDS):
    for k in range(n_folds):
        train_start = k * test_days
        train_end   = train_start + train_days
        test_start  = train_end + embargo_days
        test_end    = test_start + test_days

        if test_end > n_samples:
            break

        yield (
            np.arange(train_start, train_end),
            np.arange(test_start,  test_end),
        )



In [7]:
# ─────────────────────────────────────────────
# 5. INSTANTIATE MODELS
# ─────────────────────────────────────────────

def build_models():
    return {
        "XGBoost":      XGBRegressor(**XGB_PARAMS),
        "LightGBM":     LGBMRegressor(**LGBM_PARAMS),
        "RandomForest": RandomForestRegressor(**RF_PARAMS),
    }


# ─────────────────────────────────────────────
# 6. SOFTMAX WEIGHT HELPER
# ─────────────────────────────────────────────

def softmax_weights(scores: np.ndarray, temperature: float = 0.1) -> np.ndarray:
    scores  = np.array(scores, dtype=float)
    shifted = (scores - scores.max()) / temperature
    exp_s   = np.exp(shifted)
    return exp_s / exp_s.sum()

In [8]:
dfs['ASIAN PAINTS LTD.']

,company_code,company_name,date,opening,high,low,closing,trade Volume,Return
0,22859,ASIAN PAINTS LTD.,2005-01-03,320.10,325.00,318.60,319.10,17273,-0.003124
1,22859,ASIAN PAINTS LTD.,2005-01-04,322.00,324.90,320.00,324.05,32240,0.006366
2,22859,ASIAN PAINTS LTD.,2005-01-05,324.05,324.95,320.00,321.85,30500,-0.006789
3,22859,ASIAN PAINTS LTD.,2005-01-06,327.00,327.00,302.55,309.45,38985,-0.053670
4,22859,ASIAN PAINTS LTD.,2005-01-07,310.00,322.00,309.10,320.05,17621,0.032419
...,...,...,...,...,...,...,...,...,...
5245,22859,ASIAN PAINTS LTD.,2026-02-23,2424.70,2446.60,2411.50,2429.70,634365,0.002062
5246,22859,ASIAN PAINTS LTD.,2026-02-24,2413.00,2441.50,2403.70,2413.10,499660,0.000041
5247,22859,ASIAN PAINTS LTD.,2026-02-25,2416.00,2427.90,2400.00,2416.40,526764,0.000166
5248,22859,ASIAN PAINTS LTD.,2026-02-26,2416.10,2422.20,2382.00,2394.90,588742,-0.008774


In [9]:
make_features(dfs['ASIAN PAINTS LTD.'], macro_daily)

,date,company_code,company_name,opening,high,low,closing,trade Volume,Return,ret_1,...,price_vs_ma756,vol_252,vol_ratio_sr_lr,vol_ratio_252,macro_gdp_qrt,macro_inflation_qrt,macro_gdp_ann,macro_inflation_ann,fwd_ret_1y,fwd_ret_3y
0,2005-01-06,22859,ASIAN PAINTS LTD.,327.0,327.00,302.55,309.45,38985,-0.053670,-0.006789,...,0.000570,0.006789,1.000000,1.143564,3.916985,0.254453,7.923431,4.246344,-0.166287,-0.532212
1,2005-01-07,22859,ASIAN PAINTS LTD.,310.0,322.00,309.10,320.05,17621,0.032419,-0.053670,...,-0.028758,0.026823,1.000000,1.310442,3.916985,0.254453,7.923431,4.246344,-0.201428,-0.577773
2,2005-01-10,22859,ASIAN PAINTS LTD.,323.0,324.00,317.05,321.80,17363,-0.003715,0.032419,...,0.003606,0.031244,1.000000,0.644896,3.916985,0.254453,7.923431,4.246344,-0.223381,-0.581135
3,2005-01-11,22859,ASIAN PAINTS LTD.,318.1,321.95,318.05,320.30,24322,0.006916,-0.003715,...,0.007567,0.027950,1.000000,0.676560,3.916985,0.254453,7.923431,4.246344,-0.234533,-0.581412
4,2005-01-12,22859,ASIAN PAINTS LTD.,320.0,320.00,312.00,316.90,14280,-0.009688,0.006916,...,0.002459,0.025894,1.000000,0.954852,3.916985,0.254453,7.923431,4.246344,-0.228622,-0.584067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5242,2026-02-23,22859,ASIAN PAINTS LTD.,2424.7,2446.60,2411.50,2429.70,634365,0.002062,0.020982,...,-0.133564,0.013264,1.331556,0.657100,12.136078,-0.024015,6.494766,4.953036,NaN,NaN
5243,2026-02-24,22859,ASIAN PAINTS LTD.,2413.0,2441.50,2403.70,2413.10,499660,0.000041,0.002062,...,-0.132861,0.013263,1.405211,0.528729,12.136078,-0.024015,6.494766,4.953036,NaN,NaN
5244,2026-02-25,22859,ASIAN PAINTS LTD.,2416.0,2427.90,2400.00,2416.40,526764,0.000166,0.000041,...,-0.138645,0.013253,1.310346,0.417040,12.136078,-0.024015,6.494766,4.953036,NaN,NaN
5245,2026-02-26,22859,ASIAN PAINTS LTD.,2416.1,2422.20,2382.00,2394.90,588742,-0.008774,0.000166,...,-0.137324,0.013249,1.310714,0.439967,12.136078,-0.024015,6.494766,4.953036,NaN,NaN


In [10]:
make_features(dfs['CIPLA LTD.'],macro_daily)

,date,company_code,company_name,opening,high,low,closing,trade Volume,Return,ret_1,...,price_vs_ma756,vol_252,vol_ratio_sr_lr,vol_ratio_252,macro_gdp_qrt,macro_inflation_qrt,macro_gdp_ann,macro_inflation_ann,fwd_ret_1y,fwd_ret_3y
0,2005-01-06,47858,CIPLA LTD.,302.00,302.7,289.05,292.65,1299361,-0.030960,-0.052159,...,-0.037805,0.026680,1.000000,1.208557,3.916985,0.254453,7.923431,4.246344,-0.597807,-0.893278
1,2005-01-07,47858,CIPLA LTD.,293.00,295.9,289.10,291.75,927534,-0.004266,-0.030960,...,-0.048563,0.022281,1.000000,1.184090,3.916985,0.254453,7.923431,4.246344,-0.592145,-0.896871
2,2005-01-10,47858,CIPLA LTD.,292.00,297.9,279.00,281.90,1325949,-0.034589,-0.004266,...,-0.041620,0.021207,1.000000,0.872245,3.916985,0.254453,7.923431,4.246344,-0.577628,-0.889695
3,2005-01-11,47858,CIPLA LTD.,282.00,284.0,265.50,268.15,1699809,-0.049113,-0.034589,...,-0.062417,0.019881,1.000000,1.197627,3.916985,0.254453,7.923431,4.246344,-0.553188,-0.880055
4,2005-01-12,47858,CIPLA LTD.,270.05,272.7,245.00,252.80,2556180,-0.063877,-0.049113,...,-0.094153,0.020762,1.000000,1.426238,3.916985,0.254453,7.923431,4.246344,-0.528843,-0.876385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5242,2026-02-23,47858,CIPLA LTD.,1335.90,1335.9,1308.80,1326.50,2435610,-0.007036,0.009181,...,-0.020207,0.011911,1.189526,1.024682,12.136078,-0.024015,6.494766,4.953036,NaN,NaN
5243,2026-02-24,47858,CIPLA LTD.,1326.00,1338.5,1316.50,1326.70,2109672,0.000528,-0.007036,...,-0.031152,0.011893,0.946026,1.605280,12.136078,-0.024015,6.494766,4.953036,NaN,NaN
5244,2026-02-25,47858,CIPLA LTD.,1326.70,1349.0,1326.70,1346.10,1018377,0.014623,0.000528,...,-0.031282,0.011870,0.927640,1.386846,12.136078,-0.024015,6.494766,4.953036,NaN,NaN
5245,2026-02-26,47858,CIPLA LTD.,1352.80,1364.9,1343.30,1358.10,2013877,0.003918,0.014623,...,-0.017408,0.011901,0.891326,0.669367,12.136078,-0.024015,6.494766,4.953036,NaN,NaN


In [11]:
# ─────────────────────────────────────────────
# 7. ROLLING-WINDOW CV + ENSEMBLE (Live Tail Inference)
# ─────────────────────────────────────────────

cv_summary       = {}
backtest_results = {}
pred_returns     = {"1Y": {}, "3Y": {}}

print(f"\n{'═'*110}")
print("  ENSEMBLE ROLLING-WINDOW CV  —  DUAL HORIZON: 1-Year & 3-Year Forward Returns")
print(f"  Training : {TRAIN_DAYS} days (~5 yrs)  │  Test : {TEST_DAYS} days (~1 yr)")
print(f"{'═'*110}\n")

for name in names_list:
    short = name.replace(" LTD.", "").strip()
    d     = make_features(dfs[name], macro_daily)
    X     = d[FEATURES].values
    dates = d["date"].values

    print(f"\n  {'▶'*2} {short}  (Data length: {len(d)} rows | {d['date'].iloc[0].date()} → {d['date'].iloc[-1].date()})")

    results_by_horizon = {}

    for horizon_label, target_col, horizon_days, rf_threshold in [
        ("1Y", "fwd_ret_1y", HORIZON_1Y, BUY_THRESHOLD_1Y),
        ("3Y", "fwd_ret_3y", HORIZON_3Y, BUY_THRESHOLD_3Y * 3),
    ]:
        y = d[target_col].values

        print(f"\n  ── Horizon : {horizon_label} Forward Return ───────────────────────")

        fold_metrics    = []
        model_hit_rates = {m: [] for m in MODEL_NAMES}

        # Pass horizon_days as the embargo to purge the overlap!
        for fold_num, (tr_idx, te_idx) in enumerate(rolling_window_folds(len(d), embargo_days=horizon_days), start=1):
            X_tr, y_tr = X[tr_idx], y[tr_idx]
            X_te, y_te = X[te_idx], y[te_idx]

            # Skip folds that extend into the missing-target tail
            if np.isnan(y_tr).any() or np.isnan(y_te).any():
                continue

            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X_tr)
            X_te_s = scaler.transform(X_te)

            models = build_models()
            preds  = {}
            hits   = {}

            for mname, model in models.items():
                model.fit(X_tr_s, y_tr)
                p = model.predict(X_te_s)
                preds[mname] = p
                hits[mname] = float(np.mean((p > rf_threshold) == (y_te > rf_threshold)))
                model_hit_rates[mname].append(hits[mname])

            ens_pred = np.mean([preds[m] for m in MODEL_NAMES], axis=0)
            ens_hit  = float(np.mean((ens_pred > rf_threshold) == (y_te > rf_threshold)))
            ens_rmse = float(np.sqrt(mean_squared_error(y_te, ens_pred)))
            ens_r2   = float(r2_score(y_te, ens_pred))

            test_start_dt = pd.Timestamp(dates[te_idx[0]]).date()
            test_end_dt   = pd.Timestamp(dates[te_idx[-1]]).date()

            print(f"  Fold {fold_num:<2} {str(test_start_dt)+' → '+str(test_end_dt):<26} "
                  f"Ens Hit: {ens_hit:>7.1%} | R²: {ens_r2:>6.4f}")

            fold_metrics.append({
                "fold": fold_num, "hits": hits, "ens_hit": ens_hit,
                "ens_rmse": ens_rmse, "ens_r2": ens_r2,
                "preds": preds, "actual": y_te, "test_dates": dates[te_idx],
            })

        if not fold_metrics:
            print(f"  ⚠  No valid folds for {horizon_label}.")
            results_by_horizon[horizon_label] = None
            continue

        mean_hits     = {m: np.mean(model_hit_rates[m]) for m in MODEL_NAMES}
        blend_weights = softmax_weights([mean_hits[m] for m in MODEL_NAMES], temperature=0.1)
        blend_w_dict  = dict(zip(MODEL_NAMES, blend_weights))

        cv_ens_hit     = np.mean([m["ens_hit"]  for m in fold_metrics])
        cv_ens_rmse    = np.mean([m["ens_rmse"] for m in fold_metrics])
        cv_ens_r2      = np.mean([m["ens_r2"]   for m in fold_metrics])
        cv_ens_hit_std = np.std ([m["ens_hit"]  for m in fold_metrics])

        # ── Final model fitting for backtest ────────────────────────────
        # Use horizon_days to dynamically purge the gap before the backtest window
        final_train_end = len(d) - BACKTEST_DAYS - horizon_days
        final_train_idx = np.arange(max(0, final_train_end - TRAIN_DAYS), final_train_end)
        final_test_idx  = np.arange(final_train_end + horizon_days, len(d))

        valid_train = [i for i in final_train_idx if not np.isnan(y[i])]
        valid_test  = [i for i in final_test_idx  if not np.isnan(y[i])]

        if len(valid_train) == 0:
            continue

        X_ftr, y_ftr = X[valid_train], y[valid_train]
        scaler_f = StandardScaler()
        X_ftr_s  = scaler_f.fit_transform(X_ftr)

        final_models = build_models()
        final_fi     = {}

        for mname, model in final_models.items():
            model.fit(X_ftr_s, y_ftr)
            if hasattr(model, "feature_importances_"):
                final_fi[mname] = dict(zip(FEATURES, model.feature_importances_))

        # ── Backtest execution (on valid test targets) ─────────────────
        bt_hit = np.nan
        final_ens = np.array([])
        y_fte = np.array([])

        if len(valid_test) > 0:
            X_fte, y_fte = X[valid_test], y[valid_test]
            X_fte_s = scaler_f.transform(X_fte)
            final_preds = {m: final_models[m].predict(X_fte_s) for m in MODEL_NAMES}
            final_ens = sum(blend_w_dict[m] * final_preds[m] for m in MODEL_NAMES)
            bt_hit = float(np.mean((final_ens > rf_threshold) == (y_fte > rf_threshold)))

        # ── LIVE INFERENCE (For Today's Allocation) ────────────────────
        X_live   = X[-1].reshape(1, -1)
        X_live_s = scaler_f.transform(X_live)

        live_preds = {m: final_models[m].predict(X_live_s)[0] for m in MODEL_NAMES}
        live_ens   = sum(blend_w_dict[m] * live_preds[m] for m in MODEL_NAMES)

        years    = 1 if horizon_label == "1Y" else 3
        pred_ann = float((1 + live_ens) ** (1 / years) - 1)

        pred_returns[horizon_label][name] = pred_ann

        print(f"  Live Signal ({horizon_label}) for {d['date'].iloc[-1].date()}: {pred_ann:.2%}")

        results_by_horizon[horizon_label] = {
            "cv_hit": cv_ens_hit, "cv_rmse": cv_ens_rmse, "cv_r2": cv_ens_r2,
            "cv_hit_std": cv_ens_hit_std, "mean_hits": mean_hits,
            "blend_weights": blend_w_dict, "fold_metrics": fold_metrics,
            "final_ens": final_ens, "y_fte": y_fte, "final_fi": final_fi,
            "pred_ann_ret": pred_ann, "bt_hit": bt_hit,
            "rf_threshold": rf_threshold, "years": years,
        }

    cv_summary[name]       = results_by_horizon
    backtest_results[name] = results_by_horizon


══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  ENSEMBLE ROLLING-WINDOW CV  —  DUAL HORIZON: 1-Year & 3-Year Forward Returns
  Training : 1260 days (~5 yrs)  │  Test : 252 days (~1 yr)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════


  ▶▶ ASIAN PAINTS  (Data length: 5247 rows | 2005-01-06 → 2026-02-27)

  ── Horizon : 1Y Forward Return ───────────────────────
  Fold 1  2011-02-07 → 2012-02-09    Ens Hit:   36.5% | R²: -1.1985
  Fold 2  2012-02-10 → 2013-02-11    Ens Hit:   52.4% | R²: -0.6263
  Fold 3  2013-02-12 → 2014-02-13    Ens Hit:   57.9% | R²: -13.7107
  Fold 4  2014-02-14 → 2015-02-27    Ens Hit:   31.0% | R²: -29.6698
  Fold 5  2015-02-28 → 2016-03-03    Ens Hit:   23.8% | R²: -11.4077
  Fold 6  2016-03-04 → 2017-03-14    Ens Hit:  100.0% | R²: 0.3166
  Fold 7  2017-03-15 → 2018-03-19    Ens Hit:   82.9% | R²: -0.5547
  Fold 8  2018-03-20 → 2019-03

In [12]:
cv_summary 

{'ASIAN PAINTS LTD.': {'1Y': {'cv_hit': np.float64(0.7460317460317459),
   'cv_rmse': np.float64(0.21571250946461554),
   'cv_r2': np.float64(-4.989096057065817),
   'cv_hit_std': np.float64(0.28757786620253056),
   'mean_hits': {'XGBoost': np.float64(0.6932234432234432),
    'LightGBM': np.float64(0.7509157509157508),
    'RandomForest': np.float64(0.7155067155067154)},
   'blend_weights': {'XGBoost': np.float64(0.24812892807462716),
    'LightGBM': np.float64(0.4418063037730408),
    'RandomForest': np.float64(0.3100647681523321)},
   'fold_metrics': [{'fold': 1,
     'hits': {'XGBoost': 0.38492063492063494,
      'LightGBM': 0.376984126984127,
      'RandomForest': 0.3531746031746032},
     'ens_hit': 0.36507936507936506,
     'ens_rmse': 0.22513713597350893,
     'ens_r2': -1.1985461656366656,
     'preds': {'XGBoost': array([-0.1044765 , -0.07489797, -0.06424744, -0.06205198, -0.06330624,
             -0.06381287, -0.07357074, -0.07105547, -0.07529718, -0.07756619,
             -0

In [13]:
backtest_results

{'ASIAN PAINTS LTD.': {'1Y': {'cv_hit': np.float64(0.7460317460317459),
   'cv_rmse': np.float64(0.21571250946461554),
   'cv_r2': np.float64(-4.989096057065817),
   'cv_hit_std': np.float64(0.28757786620253056),
   'mean_hits': {'XGBoost': np.float64(0.6932234432234432),
    'LightGBM': np.float64(0.7509157509157508),
    'RandomForest': np.float64(0.7155067155067154)},
   'blend_weights': {'XGBoost': np.float64(0.24812892807462716),
    'LightGBM': np.float64(0.4418063037730408),
    'RandomForest': np.float64(0.3100647681523321)},
   'fold_metrics': [{'fold': 1,
     'hits': {'XGBoost': 0.38492063492063494,
      'LightGBM': 0.376984126984127,
      'RandomForest': 0.3531746031746032},
     'ens_hit': 0.36507936507936506,
     'ens_rmse': 0.22513713597350893,
     'ens_r2': -1.1985461656366656,
     'preds': {'XGBoost': array([-0.1044765 , -0.07489797, -0.06424744, -0.06205198, -0.06330624,
             -0.06381287, -0.07357074, -0.07105547, -0.07529718, -0.07756619,
             -0

In [14]:
# ─────────────────────────────────────────────
# 8. PREDICTED RETURN RANKING  (Deliverable 1)
# ─────────────────────────────────────────────

print(f"\n{'═'*110}")
print("  PREDICTED RETURN RANKING  —  1-Year vs 3-Year Horizon (Live Inference)")
print(f"  BUY signal threshold : must beat risk-free rate of {RF_RATE:.1%} p.a.")
print(f"{'='*110}")
print(f"  {'Rank':<5} {'Company':<35} "
      f"{'1Y Pred Ann':>13} {'1Y Signal':>10} "
      f"{'3Y Pred Ann':>13} {'3Y Signal':>10} "
      f"{'Consensus':>14}")
print(f"  {'─'*100}")

ranked_1y = sorted(names_list, key=lambda n: pred_returns["1Y"].get(n, -99), reverse=True)

for rank, name in enumerate(ranked_1y, 1):
    short  = name.replace(" LTD.", "").strip()
    ret_1y = pred_returns["1Y"].get(name, float("nan"))
    ret_3y = pred_returns["3Y"].get(name, float("nan"))

    sig_1y = "BUY  ▲" if ret_1y > BUY_THRESHOLD_1Y else "SELL ▼"
    sig_3y = "BUY  ▲" if ret_3y > BUY_THRESHOLD_3Y else "SELL ▼"

    if   sig_1y == "BUY  ▲" and sig_3y == "BUY  ▲": consensus = "STRONG BUY"
    elif sig_1y == "SELL ▼" and sig_3y == "SELL ▼": consensus = "STRONG SELL"
    elif sig_1y == "BUY  ▲":                        consensus = "TACTICAL BUY"
    else:                                           consensus = "WAIT"

    print(f"  {rank:<5} {short:<35} {ret_1y:>13.2%} {sig_1y:>10} {ret_3y:>13.2%} {sig_3y:>10} {consensus:>14}")



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  PREDICTED RETURN RANKING  —  1-Year vs 3-Year Horizon (Live Inference)
  BUY signal threshold : must beat risk-free rate of 6.5% p.a.
  Rank  Company                               1Y Pred Ann  1Y Signal   3Y Pred Ann  3Y Signal      Consensus
  ────────────────────────────────────────────────────────────────────────────────────────────────────
  1     H D F C BANK                               -0.03%     SELL ▼        -2.47%     SELL ▼    STRONG SELL
  2     ASIAN PAINTS                               -6.85%     SELL ▼       -13.74%     SELL ▼    STRONG SELL
  3     TATA CONSULTANCY SERVICES                 -15.61%     SELL ▼        -8.14%     SELL ▼    STRONG SELL
  4     ULTRATECH CEMENT                          -30.18%     SELL ▼       -20.96%     SELL ▼    STRONG SELL
  5     CIPLA                                     -34.05%     SELL ▼       -31.92%     SELL ▼    STRONG 

In [15]:
# ─────────────────────────────────────────────
# 9. PORTFOLIO OPTIMISATION  (Max Sharpe / Cash Gate) (Deliverable 2)
# ─────────────────────────────────────────────

def run_optimisation(horizon_label, mu_raw_arr, confidence_arr, Sigma, names):
    shrinkage = np.clip((confidence_arr - 0.50) / 0.10, 0, 1)
    mu_adj    = mu_raw_arr * shrinkage
    n_total   = len(names)

    buy_mask = mu_adj > 0
    n_buy    = buy_mask.sum()

    if n_buy == 0:
        return np.zeros(n_total), RF_RATE, 0.0, 0.0, True, shrinkage, mu_adj

    elif n_buy < n_total:
        buy_idx = [i for i, b in enumerate(buy_mask) if b]
        mu_buy  = mu_adj[buy_idx]
        Sig_buy = Sigma[np.ix_(buy_idx, buy_idx)]

        def neg_sharpe_p(w):
            r_ = float(np.dot(w, mu_buy))
            v_ = float(np.sqrt(w @ Sig_buy @ w))
            return -(r_ - RF_RATE) / v_ if v_ > 0 else 0.0

        lo  = min(0.05, 1.0 / n_buy)
        opt = sco.minimize(
            neg_sharpe_p,
            np.ones(n_buy) / n_buy,
            method="SLSQP",
            bounds=[(lo, 1.0)] * n_buy,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
        )
        w_out = np.zeros(n_total)
        for rank_i, idx in enumerate(buy_idx):
            w_out[idx] = opt.x[rank_i]

    else:
        def neg_sharpe_f(w):
            r_ = float(np.dot(w, mu_adj))
            v_ = float(np.sqrt(w @ Sigma @ w))
            return -(r_ - RF_RATE) / v_ if v_ > 0 else 0.0

        opt = sco.minimize(
            neg_sharpe_f,
            np.ones(n_total) / n_total,
            method="SLSQP",
            bounds=[(0.05, 0.40)] * n_total,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
        )
        w_out = opt.x

    p_ret = float(np.dot(w_out, mu_adj))
    p_vol = float(np.sqrt(w_out @ Sigma @ w_out))
    shrp  = (p_ret - RF_RATE) / p_vol if p_vol > 0 else 0.0
    return w_out, p_ret, p_vol, shrp, False, shrinkage, mu_adj


# Covariance matrix
ret_df       = pd.DataFrame({n: dfs[n].set_index("date")["Return"] for n in names_list})
ret_df.index = pd.to_datetime(ret_df.index)
cov_data     = ret_df.dropna()
cov_matrix   = cov_data.cov() * 252
Sigma        = cov_matrix.values

opt_results = {}
for horizon_label in ["1Y", "3Y"]:
    mu_raw_arr     = np.array([pred_returns[horizon_label].get(n, 0.0) for n in names_list])
    confidence_arr = np.array([
        cv_summary[n][horizon_label]["cv_hit"]
        if cv_summary[n].get(horizon_label) is not None else 0.5
        for n in names_list
    ])
    w, pr, pv, sh, cash, shrink, mu_adj = run_optimisation(
        horizon_label, mu_raw_arr, confidence_arr, Sigma, names_list
    )
    opt_results[horizon_label] = {
        "weights": w, "port_ret": pr, "port_vol": pv,
        "sharpe": sh, "cash_mode": cash, "shrinkage": shrink,
        "mu_adj": mu_adj, "mu_raw": mu_raw_arr,
    }


print(f"\n{'═'*110}")
print("  BLENDED PORTFOLIO ALLOCATION  (50% Tactical 1Y  +  50% Strategic 3Y)")
print(f"{'='*110}")

w_1y        = opt_results["1Y"]["weights"]
w_3y        = opt_results["3Y"]["weights"]
w_blend_raw = 0.5 * w_1y + 0.5 * w_3y
w_blend     = w_blend_raw / w_blend_raw.sum() if w_blend_raw.sum() > 0 else w_blend_raw
CASH_MODE   = opt_results["1Y"]["cash_mode"] and opt_results["3Y"]["cash_mode"]

if CASH_MODE:
    print("  Both horizons recommend CASH — no equity allocation. Holding Risk-Free assets.")
else:
    print(f"  {'Company':<35} {'1Y Wt':>8} {'3Y Wt':>8} {'Blended Wt':>12}")
    print(f"  {SEP}")
    for i, name in enumerate(names_list):
        short = name.replace(" LTD.", "").strip()
        print(f"  {short:<35} {w_1y[i]:>8.1%} {w_3y[i]:>8.1%} {w_blend[i]:>12.1%}")

print(f"{'='*110}")

opt_weights = w_blend  # Ensure weights are passed to the backtest


══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  BLENDED PORTFOLIO ALLOCATION  (50% Tactical 1Y  +  50% Strategic 3Y)
  Both horizons recommend CASH — no equity allocation. Holding Risk-Free assets.


In [16]:
# ─────────────────────────────────────────────
# 10. CV DIAGNOSTIC SUMMARY  (per horizon)
# ─────────────────────────────────────────────

for horizon_label in ["1Y", "3Y"]:
    print(f"\n{'═'*110}")
    print(f"  CV DIAGNOSTIC SUMMARY  —  {horizon_label} Horizon  "
          f"(ensemble mean ± std across folds)")
    print(f"{'='*110}")
    print(f"  {'Company':<35} {'XGB CV Hit':>12} {'LGB CV Hit':>12} "
          f"{'RF CV Hit':>11} {'Ens CV Hit':>12} {'Ens R²':>8} {'Stability':>10}")
    print(f"  {SEP}")

    for name in names_list:
        h = cv_summary[name].get(horizon_label)
        if h is None:
            continue
        short = name.replace(" LTD.", "").strip()
        stab  = ("High"   if h["cv_hit_std"] < 0.04 else
                 "Medium" if h["cv_hit_std"] < 0.08 else "Low")
        print(f"  {short:<35} "
              f"{h['mean_hits']['XGBoost']:>12.1%} "
              f"{h['mean_hits']['LightGBM']:>12.1%} "
              f"{h['mean_hits']['RandomForest']:>11.1%} "
              f"{h['cv_hit']:>12.1%} "
              f"{h['cv_r2']:>8.4f} "
              f"{stab:>10}")



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  CV DIAGNOSTIC SUMMARY  —  1Y Horizon  (ensemble mean ± std across folds)
  Company                               XGB CV Hit   LGB CV Hit   RF CV Hit   Ens CV Hit   Ens R²  Stability
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  ASIAN PAINTS                               69.3%        75.1%       71.6%        74.6%  -4.9891        Low
  CIPLA                                      98.4%        98.4%       98.4%        98.4%  -4.3422     Medium
  H D F C BANK                               45.1%        44.9%       48.6%        48.4% -11.3014        Low
  I T C                                      51.1%        53.8%       61.9%        55.2% -14.4620        Low
  TATA CONSULTANCY SERVICES                  51.5%        45.8%       50.1%        46.2% -14.6477        Low
  ULTRATECH CEMENT                           5

In [17]:
# ─────────────────────────────────────────────
# 11. BLEND WEIGHT SUMMARY  (per horizon)
# ─────────────────────────────────────────────

for horizon_label in ["1Y", "3Y"]:
    print(f"\n{'═'*110}")
    print(f"  ENSEMBLE BLEND WEIGHTS  —  {horizon_label} Horizon  (CV Softmax)")
    print(f"{'='*110}")
    print(f"  {'Company':<35} {'XGBoost':>12} {'LightGBM':>12} {'RandomForest':>14}")
    print(f"  {SEP}")

    for name in names_list:
        h = cv_summary[name].get(horizon_label)
        if h is None:
            continue
        short = name.replace(" LTD.", "").strip()
        bw    = h["blend_weights"]
        print(f"  {short:<35} {bw['XGBoost']:>12.3f} {bw['LightGBM']:>12.3f} {bw['RandomForest']:>14.3f}")






══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  ENSEMBLE BLEND WEIGHTS  —  1Y Horizon  (CV Softmax)
  Company                                  XGBoost     LightGBM   RandomForest
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  ASIAN PAINTS                               0.248        0.442          0.310
  CIPLA                                      0.333        0.333          0.333
  H D F C BANK                               0.295        0.287          0.417
  I T C                                      0.190        0.248          0.561
  TATA CONSULTANCY SERVICES                  0.410        0.234          0.357
  ULTRATECH CEMENT                           0.183        0.243          0.574

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  ENSEMBLE BLEND WEIGHTS  —  3Y Horizon  (CV Softmax)
  

In [18]:
# ─────────────────────────────────────────────
# 12. BACKTEST SUMMARY (Deliverable 3 - Expected vs Realized)
# ─────────────────────────────────────────────

for horizon_label in ["1Y", "3Y"]:
    print(f"\n{'═'*110}")
    print(f"  FINAL MODEL BACKTEST SUMMARY  —  {horizon_label} Horizon (Out-of-sample)")
    print(f"{'='*110}")
    print(f"  {'Company':<35} {'Act Ann Ret':>13} {'Hit Rate':>10} {'RMSE':>10}")
    print(f"  {SEP}")

    for name in names_list:
        h = backtest_results[name].get(horizon_label)
        if h is None or len(h.get('y_fte', [])) == 0:
            continue

        short   = name.replace(" LTD.", "").strip()
        y_fte   = h["y_fte"]
        years   = h["years"]
        act_ann = float((1 + np.mean(y_fte)) ** (1 / years) - 1)

        print(f"  {short:<35} "
              f"{act_ann:>13.2%} "
              f"{h['bt_hit']:>10.1%} "
              f"{float(np.sqrt(mean_squared_error(h['y_fte'], h['final_ens']))):>10.4f}")



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  FINAL MODEL BACKTEST SUMMARY  —  1Y Horizon (Out-of-sample)
  Company                               Act Ann Ret   Hit Rate       RMSE
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  ASIAN PAINTS                               -7.89%      75.4%     0.1388
  CIPLA                                      -6.75%      87.7%     0.2673
  H D F C BANK                               14.89%       3.6%     0.2239
  I T C                                     -19.78%     100.0%     0.1910
  TATA CONSULTANCY SERVICES                 -16.12%     100.0%     0.1176
  ULTRATECH CEMENT                            0.28%      83.7%     0.3365

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  FINAL MODEL BACKTEST SUMMARY  —  3Y Horizon (Out-of-sample)
  Company            

In [19]:
# ─────────────────────────────────────────────
# 13. FEATURE IMPORTANCES  (best CV stock, both horizons)
# ─────────────────────────────────────────────
for horizon_label in ["1Y", "3Y"]:
    valid_names = [n for n in names_list if cv_summary[n].get(horizon_label) is not None]

    if not valid_names:
        continue

    best = max(valid_names, key=lambda n: cv_summary[n][horizon_label]["cv_hit"])

    print(f"\n{'═'*110}")
    print(f"  FEATURE IMPORTANCES  —  {horizon_label} Horizon  |  "
          f"{best.replace(' LTD.','').strip()}  (best CV hit rate)")
    print(f"  Short-horizon + Long-horizon + Macroeconomic features")
    print(f"  (XGBoost & LightGBM: gain-based  |  Random Forest: impurity-based)")
    print(f"{'='*110}")

    h      = backtest_results[best][horizon_label]
    fi_all = h["final_fi"]
    bw     = h["blend_weights"]

    agg_fi = {}
    for feat in FEATURES:
        agg_fi[feat] = sum(
            bw[m] * fi_all[m].get(feat, 0)
            for m in MODEL_NAMES if m in fi_all
        )

    print(f"\n  {'Rank':<6}  {'Feature':<22}  {'Type':<10}  "
          f"{'XGBoost':>10}  {'LightGBM':>10}  {'RandomForest':>13}  "
          f"{'Weighted':>10}  {'% of Total':>10}")
    print(f"  {'─'*100}")

    sorted_feats = sorted(FEATURES, key=lambda f: -agg_fi[f])
    total_fi     = sum(agg_fi.values())

    for rank, feat in enumerate(sorted_feats, 1):
        xgb_fi = fi_all.get("XGBoost",      {}).get(feat, 0)
        lgb_fi = fi_all.get("LightGBM",     {}).get(feat, 0)
        rf_fi  = fi_all.get("RandomForest", {}).get(feat, 0)
        w_fi   = agg_fi[feat]
        pct    = (w_fi / total_fi * 100) if total_fi > 0 else 0

        if   feat in MACRO_FEATURES: ftype = "Macro"
        elif feat in LONG_FEATURES:  ftype = "Long-horizon"
        else:                        ftype = "Price"

        print(f"  {rank:<6}  {feat:<22}  {ftype:<10}  "
              f"{xgb_fi:>10.4f}  {lgb_fi:>10.4f}  {rf_fi:>13.4f}  "
              f"{w_fi:>10.4f}  {pct:>9.1f}%")

    print(f"  {'─'*100}")
    print(f"  {'':<6}  {'TOTAL':<22}  {'':<10}  "
          f"{'':<10}  {'':<10}  {'':<13}  {total_fi:>10.4f}  {'100.0%':>10}")

    price_total = sum(v for f, v in agg_fi.items() if f not in MACRO_FEATURES and f not in LONG_FEATURES)
    long_total  = sum(v for f, v in agg_fi.items() if f in LONG_FEATURES)
    macro_total = sum(v for f, v in agg_fi.items() if f in MACRO_FEATURES)

    print(f"\n  Feature Group Summary:")
    print(f"  {'─'*50}")
    print(f"  {'Short-horizon Price/Volume':<35}  "
          f"{price_total:.4f}  ({price_total/total_fi*100:.1f}%)")
    print(f"  {'Long-horizon Trend/Context':<35}  "
          f"{long_total:.4f}  ({long_total/total_fi*100:.1f}%)")
    print(f"  {'Macroeconomic':<35}  "
          f"{macro_total:.4f}  ({macro_total/total_fi*100:.1f}%)")



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  FEATURE IMPORTANCES  —  1Y Horizon  |  CIPLA  (best CV hit rate)
  Short-horizon + Long-horizon + Macroeconomic features
  (XGBoost & LightGBM: gain-based  |  Random Forest: impurity-based)

  Rank    Feature                 Type           XGBoost    LightGBM   RandomForest    Weighted  % of Total
  ────────────────────────────────────────────────────────────────────────────────────────────────────
  1       vol_252                 Long-horizon      0.0240    328.0000         0.0637    109.3626       13.0%
  2       price_vs_ma252          Long-horizon      0.0769    244.0000         0.0925     81.3898        9.7%
  3       ret_20                  Price           0.0144    217.0000         0.0109     72.3417        8.6%
  4       mom_756                 Long-horizon      0.0395    185.0000         0.1716     61.7370        7.4%
  5       mom_252                 Long-horizon

In [20]:
# ─────────────────────────────────────────────
# 14. PORTFOLIO-LEVEL CUMULATIVE BACKTEST (Deliverable 3 - 2 Year Window)
# ─────────────────────────────────────────────

print(f"\n{'═'*110}")
print("  PORTFOLIO CUMULATIVE BACKTEST  (Last 2 Years)")
print("  Applies live blended weights to the most recent 504 trading days.")
if CASH_MODE:
    print("  ⚠  Cash Mode — showing hypothetical equal-weight market vs risk-free")
print(f"{'='*110}")

daily_actual_frames = {}
for name in names_list:
    d_bt = make_features(dfs[name], macro_daily)

    # We grab the last BACKTEST_DAYS (504 days = ~2 years) for the portfolio simulation
    final_test_idx  = np.arange(len(d_bt) - BACKTEST_DAYS, len(d_bt))
    test_df = d_bt.iloc[final_test_idx].copy().set_index("date")

    daily_actual_frames[name] = pd.Series(
        d_bt["Return"].values[final_test_idx],
        index=test_df.index, name=name
    )

actual_matrix = pd.DataFrame(daily_actual_frames).dropna()

if CASH_MODE:
    display_weights = np.ones(len(names_list)) / len(names_list)
    weight_label    = "Equal-Weight Market (hypothetical, NOT recommended)"
else:
    display_weights = opt_weights
    weight_label    = "Blended 1Y/3Y Optimised Portfolio"

port_actual_daily = actual_matrix.dot(display_weights)
port_monthly = pd.DataFrame({"Actual": port_actual_daily.resample("ME").sum()}).dropna()
port_monthly["Actual_Cum"] = (1 + port_monthly["Actual"]).cumprod() - 1

# Cash benchmark
monthly_rf                  = (1 + RF_RATE / 252) ** 21 - 1
port_monthly["Cash_Return"] = monthly_rf
port_monthly["Cash_Cum"]    = (1 + port_monthly["Cash_Return"]).cumprod() - 1

print(f"\n  Weights : {weight_label}")
if CASH_MODE:
    print(f"\n  {'Month':<12} {'Mkt Act Return':>16} {'Mkt Cum':>12} {'Cash Cum':>12} {'Cash Advantage':>16}")
    print(f"  {'─'*75}")
    for dt, row in port_monthly.iterrows():
        adv = row["Cash_Cum"] - row["Actual_Cum"]
        print(f"  {str(dt.date()):<12} {row['Actual']:>16.3%} {row['Actual_Cum']:>12.3%} {row['Cash_Cum']:>12.3%} {adv:>+16.3%}")
else:
    print(f"\n  {'Month':<12} {'Act Return':>12} {'Act Cumul':>12} {'Cash Cumul':>12} {'Active Alpha':>14}")
    print(f"  {'─'*70}")
    for dt, row in port_monthly.iterrows():
        alpha = row["Actual_Cum"] - row["Cash_Cum"]
        print(f"  {str(dt.date()):<12} {row['Actual']:>12.3%} {row['Actual_Cum']:>12.3%} {row['Cash_Cum']:>12.3%} {alpha:>+14.3%}")

final_actual_cum = port_monthly["Actual_Cum"].iloc[-1]
final_cash_cum   = port_monthly["Cash_Cum"].iloc[-1]
alpha_2y         = final_actual_cum - final_cash_cum

print(f"\n  {'─'*75}")
print(f"  2-Year Cumulative Return  (Portfolio) : {final_actual_cum:.2%}")
print(f"  2-Year Cumulative Return  (Cash)      : {final_cash_cum:.2%}")
print(f"  Active Alpha vs Cash                  : {alpha_2y:+.2%}")

if CASH_MODE:
    print(f"\n  ✦  By following the SELL signal and holding cash, you would have")
    print(f"     {'underperformed' if alpha_2y > 0 else 'outperformed'} the market by {abs(alpha_2y):.2%} over 2 years.")
else:
    ann_actual = (1 + final_actual_cum) ** (1 / 2) - 1
    ann_cash   = (1 + final_cash_cum)   ** (1 / 2) - 1
    print(f"  Annualised Return  (Portfolio)        : {ann_actual:.2%}")
    print(f"  Annualised Return  (Cash)             : {ann_cash:.2%}")

port_monthly.to_csv('output/backtest_portfolio.csv')
print(f"\n  → output/backtest_portfolio.csv saved.")
print(f"{'='*110}")


══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  PORTFOLIO CUMULATIVE BACKTEST  (Last 2 Years)
  Applies live blended weights to the most recent 504 trading days.
  ⚠  Cash Mode — showing hypothetical equal-weight market vs risk-free

  Weights : Equal-Weight Market (hypothetical, NOT recommended)

  Month          Mkt Act Return      Mkt Cum     Cash Cum   Cash Advantage
  ───────────────────────────────────────────────────────────────────────────
  2024-02-29            -0.441%      -0.441%       0.543%          +0.984%
  2024-03-31            -0.667%      -1.105%       1.089%          +2.194%
  2024-04-30            -2.439%      -3.517%       1.638%          +5.155%
  2024-05-31            -1.895%      -5.345%       2.190%          +7.535%
  2024-06-30             0.477%      -4.894%       2.745%          +7.639%
  2024-07-31             4.981%      -0.157%       3.303%          +3.460%
  2024-08-31             0.988% 